In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# XGBoost (설치가 안 되어 있으면 아래 주석을 해제 후 실행)
from xgboost import XGBClassifier
import warnings

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

In [ ]:
df = pd.read_csv("../../data_processed/product_type_1.csv", header=[0, 1])

In [ ]:
def get_train_test_data(product_type, defect_groups, test_size=0.2, random_state=42):

    # X 준비
    X = pd.concat([product_type['process'], product_type['sensor']], axis=1)
    X.columns = (
        [f"process_{col}" for col in product_type['process'].columns] +
        [f"sensor_{col}"  for col in product_type['sensor'].columns]
    )

    # y 준비
    defects_data = product_type['defects']
    y = pd.DataFrame(index=defects_data.index)
    for group, cols in defect_groups.items():
        usable = [c for c in cols if c in defects_data.columns]
        y[group] = defects_data[usable].max(axis=1) if usable else 0

    # stratify 기준 생성
    strata = y.astype(str).agg("".join, axis=1)

    # train/test 분리
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=strata
    )

    # 결과 출력
    print(f"Train set: {X_train.shape} / Test set: {X_test.shape}")
    print("\n[불량 유형 비율(%)]")
    display((y.mean() * 100).round(2).to_frame("rate_%"))
    print("\n[Train set 타겟 분포]")
    print(y_train.value_counts())
    print("\n[Test set 타겟 분포]")
    print(y_test.value_counts())

    return X_train, X_test, y_train, y_test

defect_groups = {
    "표면": [
        "exfoliation_1",
        "stain_1",
        "dent_1",
        "exfoliation_2"
    ],

    "구조": [
        "short_shot_1",
        "bubble_1",
        "short_shot_2",
        "bubble_2",
        "deformation_1",
        "deformation_2"
    ]
}

X_train, X_test, y_train, y_test = get_train_test_data(df, defect_groups, test_size=0.2, random_state=42)

In [ ]:
# ── Model 1: 불량 여부 이진 분류 ──────────────────────────────
# y: 표면 또는 구조 중 하나라도 불량이면 1
y_train_bin = (y_train.max(axis=1)).astype(int)
y_test_bin  = (y_test.max(axis=1)).astype(int)

model_defect = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=(y_train_bin == 0).sum() / (y_train_bin == 1).sum(),
    random_state=42,
    eval_metric='logloss',
)
model_defect.fit(X_train, y_train_bin)

pred_bin = model_defect.predict(X_test)
prob_bin = model_defect.predict_proba(X_test)[:, 1]

print('=' * 60)
print('Model 1 — 불량 여부 이진 분류')
print('=' * 60)
print(classification_report(y_test_bin, pred_bin, target_names=['정상', '불량']))
print(f'ROC-AUC : {roc_auc_score(y_test_bin, prob_bin):.4f}')
print(f'PR-AUC  : {average_precision_score(y_test_bin, prob_bin):.4f}')


# ── Model 2: 불량 유형 다중 분류 ──────────────────────────────
# 0: 정상 / 1: 표면만 / 2: 구조만 / 3: 복합(표면+구조)
def encode_defect_type(y):
    return (y['표면'] * 1 + y['구조'] * 2).clip(upper=3).astype(int)

y_train_mc = encode_defect_type(y_train)
y_test_mc  = encode_defect_type(y_test)

model_type = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    objective='multi:softmax',
    num_class=4,
    random_state=42,
    eval_metric='mlogloss',
)
model_type.fit(X_train, y_train_mc)

pred_mc = model_type.predict(X_test)

print()
print('=' * 60)
print('Model 2 — 불량 유형 다중 분류 (0=정상 / 1=표면 / 2=구조 / 3=복합)')
print('=' * 60)
print(classification_report(y_test_mc, pred_mc, target_names=['정상', '표면', '구조', '복합']))